# Deteksi Gambar Asli vs Buatan AI (CIFAKE) — Training di Google Colab

Notebook ini akan:
1. Menulis ulang seluruh source code proyek (`src/`) langsung di sini (tidak perlu upload file terpisah).
2. Download dataset **CIFAKE** dari Kaggle.
3. Training CNN ringan, evaluasi, dan export ke TensorFlow Lite.
4. Download hasilnya (model + laporan) ke laptopmu.

**Referensi:**
- Bird, J.J. & Lotfi, A. (2024). *CIFAKE: Image Classification and Explainable Identification of AI-Generated Synthetic Images*. IEEE Access.
- Lokner Ladjevic, A., Kramberger, T., Kramberger, R., & Vlahek, D. (2024). *Detection of AI-Generated Synthetic Images with a Lightweight CNN*. AI (MDPI), 5(3), 1575–1593.

---
### Langkah 0 — Aktifkan GPU (penting, biar training cepat!)
Klik menu **Runtime > Change runtime type > Hardware accelerator > GPU (T4)** lalu **Save**,
baru jalankan cell-cell di bawah ini dari atas ke bawah.


In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU terdeteksi:", gpus)
else:
    print("⚠️  Tidak ada GPU terdeteksi. Buka menu Runtime > Change runtime type > GPU, lalu jalankan ulang cell ini.")


---
### Langkah 1 — Install dependency tambahan
TensorFlow, scikit-learn, matplotlib, Pillow biasanya sudah tersedia bawaan di Colab.
Kita cuma perlu menambah `kaggle` (untuk download dataset).


In [ ]:
!pip install -q kaggle scikit-learn pillow matplotlib
print("Selesai install dependency.")


---
### Langkah 2 — Tulis ulang source code proyek

Cell-cell di bawah ini membuat struktur folder `src/` dan mengisi setiap file
dengan kode yang sama persis dengan yang sudah saya test di proyek `cifake-detector`.


In [ ]:
import os
for d in ["src", "app", "data", "models", "reports"]:
    os.makedirs(d, exist_ok=True)
print("Struktur folder dibuat:", os.listdir("."))


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""
config.py
Semua konstanta & hyperparameter proyek ada di sini, supaya kalau mau
ganti setting (ukuran gambar, batch size, dll) cukup edit di satu tempat.
"""

import os

# ---- Path dataset ----
# Struktur folder yang diharapkan setelah kamu download & extract CIFAKE dari Kaggle:
#
# data/
#   train/
#     REAL/   <- ribuan file .jpg foto asli (dari CIFAR-10)
#     FAKE/   <- ribuan file .jpg foto hasil AI (Stable Diffusion)
#   test/
#     REAL/
#     FAKE/
#
# Kalau nama foldermu beda (misal "real"/"fake" huruf kecil), tinggal
# sesuaikan TRAIN_DIR/TEST_DIR atau rename foldernya.

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

KERAS_MODEL_PATH = os.path.join(MODELS_DIR, "cifake_cnn.keras")
BEST_CHECKPOINT_PATH = os.path.join(MODELS_DIR, "cifake_cnn_best.keras")
TFLITE_MODEL_PATH = os.path.join(MODELS_DIR, "cifake_cnn.tflite")
TFLITE_QUANT_MODEL_PATH = os.path.join(MODELS_DIR, "cifake_cnn_quant.tflite")

# ---- Gambar & data ----
# CIFAKE aslinya 32x32 (mengikuti resolusi CIFAR-10). Kita pertahankan
# 32x32 supaya modelnya tetap ringan & cepat dilatih di CPU.
IMG_SIZE = (32, 32)
CHANNELS = 3
BATCH_SIZE = 64
SEED = 42

# Nama kelas -> index. image_dataset_from_directory akan mengurutkan
# folder secara alfabetis, jadi FAKE=0, REAL=1.
CLASS_NAMES = ["FAKE", "REAL"]

# ---- Training ----
EPOCHS = 30
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 5
VALIDATION_SPLIT = 0.1  # diambil dari train/ untuk validasi selama training


In [ ]:
%%writefile src/dataset.py
"""
dataset.py
Memuat gambar dari folder train/ dan test/ menjadi tf.data.Dataset yang
siap dipakai untuk training. Termasuk normalisasi pixel dan augmentasi
data ringan (biar model tidak gampang overfit).
"""

import tensorflow as tf
from tensorflow.keras import layers

from . import config


def _prepare(ds: tf.data.Dataset, augment: bool, normalize: layers.Layer) -> tf.data.Dataset:
    """Normalisasi pixel ke [0,1], augmentasi (opsional), lalu prefetch."""
    ds = ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.05),
        ])
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)


def load_datasets():
    """
    Return: (train_ds, val_ds, test_ds)
    Label mengikuti urutan alfabetis nama folder -> FAKE=0, REAL=1
    (sudah didefinisikan juga di config.CLASS_NAMES).
    """
    normalize = layers.Rescaling(1.0 / 255)

    train_ds = tf.keras.utils.image_dataset_from_directory(
        config.TRAIN_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        seed=config.SEED,
        validation_split=config.VALIDATION_SPLIT,
        subset="training",
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        config.TRAIN_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        seed=config.SEED,
        validation_split=config.VALIDATION_SPLIT,
        subset="validation",
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        config.TEST_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        shuffle=False,  # penting: urutan dijaga supaya cocok dengan y_true saat evaluasi
    )

    train_ds = _prepare(train_ds, augment=True, normalize=normalize)
    val_ds = _prepare(val_ds, augment=False, normalize=normalize)
    test_ds = _prepare(test_ds, augment=False, normalize=normalize)

    return train_ds, val_ds, test_ds


In [ ]:
%%writefile src/model.py
"""
model.py
Arsitektur CNN ringan untuk klasifikasi biner REAL vs FAKE.

Desainnya terinspirasi dari pendekatan "lightweight CNN" yang dipakai
Lokner Ladjevic dkk. (2024, MDPI AI 5(3):1575-1593) untuk kasus yang
sama persis (deteksi gambar AI-generated di dataset CIFAKE): total
8 convolutional layer + 2 hidden (dense) layer. Catatan: paper tsb
tidak mempublikasikan tabel hyperparameter super detail (ukuran
kernel/filter tiap layer) di bagian yang bisa saya akses, jadi
implementasi berikut adalah desain umum & wajar yang mengikuti pola
tersebut (4 blok, tiap blok 2 conv layer), bukan replikasi 1:1 dari
kode aslinya. Jelaskan hal ini juga di laporan/skripsimu supaya jujur
soal mana yang "mengikuti ide paper" vs "hasil desain sendiri".

Total parameter model ini sekitar 300rb-600rb (tergantung filter),
jauh lebih kecil dari ResNet/VGG (puluhan juta parameter) -> cocok
disebut "algoritma ringan" di laporan kamu.
"""

from tensorflow.keras import layers, models

from . import config


def build_lightweight_cnn(input_shape=(32, 32, 3)) -> models.Model:
    inputs = layers.Input(shape=input_shape, name="image")

    x = inputs
    # 4 blok, masing-masing 2 conv layer -> total 8 conv layer
    filters_per_block = [32, 64, 128, 128]
    for i, f in enumerate(filters_per_block):
        x = layers.Conv2D(f, 3, padding="same", activation=None, name=f"block{i+1}_conv1")(x)
        x = layers.BatchNormalization(name=f"block{i+1}_bn1")(x)
        x = layers.Activation("relu", name=f"block{i+1}_act1")(x)

        x = layers.Conv2D(f, 3, padding="same", activation=None, name=f"block{i+1}_conv2")(x)
        x = layers.BatchNormalization(name=f"block{i+1}_bn2")(x)
        x = layers.Activation("relu", name=f"block{i+1}_act2")(x)

        # Jangan pooling lagi kalau feature map sudah kecil (image 32x32 -> setelah
        # 3x pooling jadi 4x4, blok ke-4 kita biarkan tanpa pooling tambahan)
        if i < 3:
            x = layers.MaxPooling2D(2, name=f"block{i+1}_pool")(x)
        x = layers.Dropout(0.25, name=f"block{i+1}_drop")(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # 2 hidden (dense) layer
    x = layers.Dense(128, activation="relu", name="hidden1")(x)
    x = layers.Dropout(0.5, name="hidden1_drop")(x)
    x = layers.Dense(64, activation="relu", name="hidden2")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)

    model = models.Model(inputs, outputs, name="lightweight_cifake_cnn")
    return model


def compile_model(model: models.Model) -> models.Model:
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            "AUC",
            "Precision",
            "Recall",
        ],
    )
    return model


if __name__ == "__main__":
    m = build_lightweight_cnn(input_shape=(*config.IMG_SIZE, config.CHANNELS))
    m = compile_model(m)
    m.summary()


In [ ]:
%%writefile src/train.py
"""
train.py
Jalankan training dari root proyek dengan:

    python -m src.train

Akan menghasilkan:
  - models/cifake_cnn_best.keras   (checkpoint terbaik berdasarkan val_loss)
  - models/cifake_cnn.keras        (model di epoch terakhir)
  - reports/training_history.png   (grafik akurasi & loss per epoch)
  - reports/training_history.csv   (angka mentahnya, buat lampiran laporan)
"""

import os
import csv

import tensorflow as tf
import matplotlib
matplotlib.use("Agg")  # supaya bisa jalan tanpa display (headless/server)
import matplotlib.pyplot as plt

from . import config
from .dataset import load_datasets
from .model import build_lightweight_cnn, compile_model


def plot_history(history, out_path):
    hist = history.history
    epochs_range = range(1, len(hist["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(epochs_range, hist["loss"], label="train_loss")
    axes[0].plot(epochs_range, hist["val_loss"], label="val_loss")
    axes[0].set_title("Loss per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs_range, hist["accuracy"], label="train_accuracy")
    axes[1].plot(epochs_range, hist["val_accuracy"], label="val_accuracy")
    axes[1].set_title("Akurasi per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def save_history_csv(history, out_path):
    hist = history.history
    keys = list(hist.keys())
    n_epochs = len(hist[keys[0]])
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch"] + keys)
        for i in range(n_epochs):
            writer.writerow([i + 1] + [hist[k][i] for k in keys])


def main():
    os.makedirs(config.MODELS_DIR, exist_ok=True)
    os.makedirs(config.REPORTS_DIR, exist_ok=True)

    print(">> Memuat dataset ...")
    train_ds, val_ds, test_ds = load_datasets()

    print(">> Membangun model ...")
    model = build_lightweight_cnn(input_shape=(*config.IMG_SIZE, config.CHANNELS))
    model = compile_model(model)
    model.summary()

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            config.BEST_CHECKPOINT_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=config.EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    print(">> Mulai training ...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.EPOCHS,
        callbacks=callbacks,
    )

    model.save(config.KERAS_MODEL_PATH)
    print(f">> Model tersimpan di: {config.KERAS_MODEL_PATH}")
    print(f">> Checkpoint terbaik di: {config.BEST_CHECKPOINT_PATH}")

    plot_history(history, os.path.join(config.REPORTS_DIR, "training_history.png"))
    save_history_csv(history, os.path.join(config.REPORTS_DIR, "training_history.csv"))
    print(">> Grafik & log training tersimpan di folder reports/")

    print(">> Evaluasi cepat di test set ...")
    results = model.evaluate(test_ds, return_dict=True)
    for k, v in results.items():
        print(f"   {k}: {v:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py
Evaluasi model yang sudah dilatih terhadap test set: accuracy, precision,
recall, F1-score, confusion matrix (gambar), dan classification report
(teks). Semua hasil disimpan di folder reports/ supaya gampang ditempel
ke laporan/skripsi.

Jalankan dari root proyek:
    python -m src.evaluate
"""

import os

import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)

from . import config
from .dataset import load_datasets


def main(model_path: str = None):
    model_path = model_path or config.BEST_CHECKPOINT_PATH
    print(f">> Memuat model dari: {model_path}")
    model = tf.keras.models.load_model(model_path)

    print(">> Memuat test set ...")
    _, _, test_ds = load_datasets()

    y_true = []
    y_prob = []
    for batch_x, batch_y in test_ds:
        preds = model.predict(batch_x, verbose=0).flatten()
        y_prob.extend(preds.tolist())
        y_true.extend(batch_y.numpy().flatten().tolist())

    y_true = np.array(y_true, dtype=int)
    y_prob = np.array(y_prob, dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)

    os.makedirs(config.REPORTS_DIR, exist_ok=True)

    report = classification_report(
        y_true, y_pred, target_names=config.CLASS_NAMES, digits=4
    )
    print(report)

    auc = roc_auc_score(y_true, y_prob)
    print(f"ROC-AUC: {auc:.4f}")

    with open(os.path.join(config.REPORTS_DIR, "classification_report.txt"), "w") as f:
        f.write(report)
        f.write(f"\nROC-AUC: {auc:.4f}\n")

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=config.CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix - CIFAKE Test Set")
    fig.tight_layout()
    fig.savefig(os.path.join(config.REPORTS_DIR, "confusion_matrix.png"), dpi=150)
    plt.close(fig)

    print(">> classification_report.txt & confusion_matrix.png tersimpan di reports/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/export_tflite.py
"""
export_tflite.py
Convert model .keras -> .tflite supaya bisa dipasang di aplikasi mobile
(Android/iOS) atau di alat yang resource-nya terbatas.

Menghasilkan 2 versi:
  1. cifake_cnn.tflite        -> float32 biasa, tanpa kompresi tambahan
  2. cifake_cnn_quant.tflite  -> dynamic range quantization (bobot di
     kompres ke int8), ukurannya jauh lebih kecil & inferensinya lebih
     cepat di HP, dengan penurunan akurasi yang biasanya kecil.

Jalankan dari root proyek:
    python -m src.export_tflite
"""

import os

import tensorflow as tf

from . import config


def convert(model_path: str, out_path: str, quantize: bool):
    model = tf.keras.models.load_model(model_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    size_kb = os.path.getsize(out_path) / 1024
    print(f">> Tersimpan: {out_path}  ({size_kb:.1f} KB)")
    return size_kb


def main():
    os.makedirs(config.MODELS_DIR, exist_ok=True)

    keras_size_kb = os.path.getsize(config.BEST_CHECKPOINT_PATH) / 1024
    print(f"Ukuran model Keras asli: {keras_size_kb:.1f} KB")

    plain_kb = convert(config.BEST_CHECKPOINT_PATH, config.TFLITE_MODEL_PATH, quantize=False)
    quant_kb = convert(config.BEST_CHECKPOINT_PATH, config.TFLITE_QUANT_MODEL_PATH, quantize=True)

    print("\nRingkasan ukuran model:")
    print(f"  Keras (.keras)            : {keras_size_kb:8.1f} KB")
    print(f"  TFLite float32 (.tflite)  : {plain_kb:8.1f} KB")
    print(f"  TFLite quantized (.tflite): {quant_kb:8.1f} KB")
    print("\nCatatan: setelah kuantisasi, cek ulang akurasinya di test set")
    print("(lihat predict.py / evaluate.py) sebelum dipasang ke app -")
    print("kadang ada sedikit penurunan akurasi yang perlu dicek dulu.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/predict.py
"""
predict.py
Coba model ke satu file gambar dari command line. Berguna buat demo cepat
sebelum bikin UI (Streamlit/mobile app).

Contoh pakai model Keras:
    python -m src.predict --image contoh.jpg

Contoh pakai model TFLite (yang nanti dipasang ke HP):
    python -m src.predict --image contoh.jpg --tflite models/cifake_cnn_quant.tflite
"""

import argparse

import numpy as np
import tensorflow as tf
from PIL import Image

from . import config


def load_image(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize(config.IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr


def predict_keras(image_arr: np.ndarray, model_path: str) -> float:
    model = tf.keras.models.load_model(model_path)
    batch = np.expand_dims(image_arr, axis=0)
    prob = model.predict(batch, verbose=0)[0][0]
    return float(prob)


def predict_tflite(image_arr: np.ndarray, tflite_path: str) -> float:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    batch = np.expand_dims(image_arr, axis=0).astype(input_details[0]["dtype"])
    interpreter.set_tensor(input_details[0]["index"], batch)
    interpreter.invoke()
    prob = interpreter.get_tensor(output_details[0]["index"])[0][0]
    return float(prob)


def interpret(prob_real: float) -> str:
    # Ingat: label index 1 = REAL, 0 = FAKE (lihat config.CLASS_NAMES)
    label = "REAL (foto asli)" if prob_real >= 0.5 else "FAKE (dibuat AI)"
    confidence = prob_real if prob_real >= 0.5 else 1 - prob_real
    return f"{label}  |  confidence: {confidence * 100:.1f}%"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", required=True, help="path ke file gambar")
    parser.add_argument("--model", default=config.BEST_CHECKPOINT_PATH,
                         help="path ke model .keras (dipakai kalau --tflite tidak diisi)")
    parser.add_argument("--tflite", default=None,
                         help="path ke model .tflite (opsional, override --model)")
    args = parser.parse_args()

    image_arr = load_image(args.image)

    if args.tflite:
        prob_real = predict_tflite(image_arr, args.tflite)
    else:
        prob_real = predict_keras(image_arr, args.model)

    print(interpret(prob_real))


if __name__ == "__main__":
    main()


> `src/config.py` sudah diset supaya path dataset/model relatif terhadap root
> proyek (folder tempat notebook ini jalan), jadi tidak perlu diedit khusus untuk Colab.


---
### Langkah 3 — Download dataset CIFAKE dari Kaggle

1. Buka https://www.kaggle.com/settings/account → bagian **API** → klik **Create New Token**.
   File `kaggle.json` akan otomatis terdownload ke laptopmu.
2. Jalankan cell di bawah, lalu upload file `kaggle.json` tadi saat diminta.


In [ ]:
from google.colab import files
print("Upload file kaggle.json kamu di sini:")
uploaded = files.upload()  # pilih file kaggle.json

import os
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(list(uploaded.values())[0])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("kaggle.json terpasang.")


In [ ]:
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p data --unzip
print("Download & extract selesai.")


In [ ]:
# Cek struktur folder hasil extract -- PENTING supaya path cocok dengan src/config.py
import os

for root, dirs, files in os.walk("data"):
    depth = root.replace("data", "").count(os.sep)
    if depth <= 2:
        n_files = len(files)
        print(f"{root}  ->  {n_files} file" if n_files else root)


Kalau hasil `os.walk` di atas menunjukkan struktur yang **berbeda** dari
`data/train/REAL`, `data/train/FAKE`, `data/test/REAL`, `data/test/FAKE`
(misalnya ada folder pembungkus tambahan), jalankan cell berikut untuk
merapikannya secara otomatis. Kalau strukturnya sudah pas, cell ini aman
dilewati/tidak akan mengubah apa-apa.


In [ ]:
import shutil, glob

def find_and_flatten(class_name, split):
    # Cari folder bernama REAL/FAKE (case-insensitive) di dalam data/, lalu
    # pindahkan ke data/<split>/<class_name> kalau lokasinya belum pas.
    target = f"data/{split}/{class_name}"
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        return  # sudah pas, tidak perlu apa-apa

    candidates = glob.glob(f"data/**/{class_name.lower()}", recursive=True) + \
                 glob.glob(f"data/**/{class_name}", recursive=True)
    candidates = [c for c in set(candidates) if split.lower() in c.lower()]

    if candidates:
        os.makedirs(f"data/{split}", exist_ok=True)
        if os.path.abspath(candidates[0]) != os.path.abspath(target):
            shutil.move(candidates[0], target)
            print(f"Dipindahkan: {candidates[0]} -> {target}")
    else:
        print(f"⚠️  Tidak ketemu folder untuk {split}/{class_name}, cek manual dengan `!find data -type d`")

for split in ["train", "test"]:
    for cls in ["REAL", "FAKE"]:
        find_and_flatten(cls, split)

print("\nStruktur akhir:")
!find data -maxdepth 2 -type d | sort


---
### Langkah 4 — Training

Dengan GPU T4 di Colab, training dataset penuh (100rb gambar, 32x32 px)
biasanya jauh lebih cepat daripada di CPU laptop biasa.


In [ ]:
!python -m src.train


Lihat grafik loss & akurasi per epoch:

In [ ]:
from IPython.display import Image, display
display(Image("reports/training_history.png"))


---
### Langkah 5 — Evaluasi (precision, recall, F1, confusion matrix)


In [ ]:
!python -m src.evaluate


In [ ]:
from IPython.display import Image, display
display(Image("reports/confusion_matrix.png"))

with open("reports/classification_report.txt") as f:
    print(f.read())


---
### Langkah 6 — Export ke TensorFlow Lite (untuk mobile)


In [ ]:
!python -m src.export_tflite


---
### Langkah 7 — Download hasil ke laptopmu

Runtime Colab bersifat sementara — begitu sesi berakhir, semua file di sini
akan hilang. **Jangan lupa download hasilnya** sebelum menutup tab/keluar.


In [ ]:
import shutil
shutil.make_archive("hasil_training_cifake", "zip", ".", "models")
shutil.make_archive("hasil_laporan_cifake", "zip", ".", "reports")

from google.colab import files
files.download("hasil_training_cifake.zip")
files.download("hasil_laporan_cifake.zip")


**Opsional — simpan ke Google Drive** (kalau mau lanjut training lagi nanti
tanpa perlu download ulang dataset dari Kaggle tiap sesi):

```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cifake-detector
!cp -r data models reports src /content/drive/MyDrive/cifake-detector/
```

---
### Catatan untuk laporan tugas akhir
- Arsitektur CNN di `src/model.py` mengikuti *pola* (8 conv layer + 2 dense
  layer) dari paper Lokner Ladjevic dkk. (2024), bukan replikasi
  hyperparameter 1:1 — sebutkan ini di bab metodologi.
- Kalau mau lanjut coba training di HP-mu sendiri (bukan Colab), file
  `.tflite` hasil Langkah 6 tinggal diintegrasikan ke aplikasi Android/iOS,
  atau dicoba dulu lewat `src/predict.py --tflite ...` / demo Streamlit di
  `app/demo_streamlit.py`.
